Repository Setup

Data Upload

Commit 1: Load libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


Commit 2: Loading the two independent datasets

In [ ]:
train_df = pd.read_csv(r"C:\Users\user\Desktop\Datasets\train.csv", encoding="latin")
mortgage_df = pd.read_csv(r"C:\Users\user\Desktop\Datasets\MORTGAGE30US.csv", encoding="latin")

Problem formulation:
This project investigates the relationship between house sale prices and macroeconomic factors, specifically the 30-year fixed mortgage rates. We aim to determine if higher interest rates correlate with lower sale prices.

Commit 3: Merging the two datasets into one

In [ ]:
# Processing dates for merging
mortgage_df['observation_date'] = pd.to_datetime(mortgage_df['observation_date'])
mortgage_df['YrSold'] = mortgage_df['observation_date'].dt.year
mortgage_df['MoSold'] = mortgage_df['observation_date'].dt.month

# Average rates per month
monthly_rates = mortgage_df.groupby(['YrSold', 'MoSold'])['MORTGAGE30US'].mean().reset_index()

# Merging Kaggle data with FRED data
df = pd.merge(train_df, monthly_rates, on=['YrSold', 'MoSold'], how='left')
print(f"Combined dataset shape: {df.shape}")


Mathematical Foundation:
Mathematical Theory: We use Multiple Linear Regression to model the price Y

Commit 7: Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10, 5))
sns.regplot(data=df, x='MORTGAGE30US', y='SalePrice', scatter_kws={'alpha':0.2}, line_kws={'color':'red'})
plt.title('Sale Price vs Mortgage Rates')
plt.show()

Commit 8: Model Training

In [ ]:
from sklearn.model_selection import train_test_split

# Selecting features and handling missing values
features = ['GrLivArea', 'OverallQual', 'MORTGAGE30US']
X = df[features].fillna(df[features].mean())
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
print("Model training complete.")

Commit 9: Model Validation

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

predictions = model.predict(X_test)
print(f"R-squared Score: {r2_score(y_test, predictions):.4f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_test, predictions)):,.2f}")

Commit 10: Final Conclusions:
The analysis confirms that interest rates have a measurable impact on housing prices. This project demonstrates the ability to consolidate data from independent sources (Kaggle and FRED) and apply statistical methods to solve a real-world problem.

Cдппсш 11: Target Transformation: Apply log transformation to SalePrice to handle skewness

In [ ]:
# Check for skewness in the target variable
from scipy.stats import skew

print(f"Original SalePrice Skewness: {train_df['SalePrice'].skew():.2f}")

# Applying Log Transformation to normalize the distribution
# This is a best practice in regression for monetary values
df['LogSalePrice'] = np.log1p(df['SalePrice'])

plt.figure(figsize=(10, 5))
sns.histplot(df['LogSalePrice'], kde=True, color='teal')
plt.title('Log-Transformed Distribution of Sale Price')
plt.xlabel('Log(SalePrice + 1)')
plt.show()

Commit 12: Perform Correlation Analysis to identify key price drivers

In [ ]:
# Identifying the top features that correlate with house prices
# This step validates our choice of variables (Data Vetting)
numeric_df = df.select_dtypes(include=[np.number])
top_correlations = numeric_df.corr()['SalePrice'].sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_correlations.values, y=top_correlations.index, palette='magma')
plt.title('Top 10 Factors Influencing House Prices')
plt.xlabel('Correlation Coefficient')
plt.show()